# Biogeme: six-mode choice calibration — student Jupyter exercise

**Designed for:** Google Colab, a hosted Jupyter Notebook or JupyterLab with a **server-backed Python 3.12+ kernel**. No IDE or special local file paths are required.

**Files students need:** Put `Biogeme_6Mode_Student_Jupyter_Same_Folder.ipynb` and `SP6ModeChoiceTLV.xlsx` **in the same Jupyter working folder**, and open the notebook there. The workbook is read directly from the accompanying file; it is **not embedded** in the notebook. Your instructor may also supply `CaseStudy1_MNL_TLV.docx` for reference.

**To run:** Select **Run → Run All Cells** (JupyterLab) or **Runtime → Run all** (Colab). The first code cell checks the Python environment and installs missing packages when permitted. In Google Colab, upload the Excel workbook to the Colab session's `/content` working directory using the Files sidebar before running all cells. A workbook stored only on your local computer or in a different Google Drive folder cannot be read until it is uploaded or mounted.

**Not supported:** browser-only **JupyterLite / Pyodide** kernels, which may not support Biogeme's compiled scientific dependencies. Use Colab or a full Python-backed hosted Jupyter service.

### What you will estimate

For SP task $t$ from traveller $n$ and alternative $i$:

$$
U_{nti}=ASC_i+\beta_{TT}\,TT_{nti}+\beta_{COST,i}\,COST_{nti}
       +\beta_{WORK2,i}\,\mathbf 1(work_n=2)
       +\beta_{WORK3,i}\,\mathbf 1(work_n=3)+\varepsilon_{nti}.
$$

- **Travel time is generic**: the same `B_TIME` appears in every mode's utility.
- **Cost and employment effects are mode-specific** in the final specification, subject to estimability.
- The reference mode is **bus with walk access (BW)**. Its ASC and person-specific coefficients are fixed to zero, so the remaining ASC and work effects are identified relative to BW.
- Employment class 1 (full-time) is the reference. The dataset does not have class-4 observations.
- Car-passenger cost is not observed separately. It is initially treated as zero, not inferred as a measured fare.
- Repeated SP tasks are grouped by respondent in the Biogeme panel likelihood.

**Exercise:** Before running the estimates, write down expected signs of the time and cost effects. Compare the three specifications, choose a defensible final model, and discuss unreliable or counterintuitive parameters. The provided data come from the six-mode Tel Aviv SP workbook and companion case-study instructions.

## 1. Notebook setup (first cell to run)

This checks whether the *Jupyter kernel* (not the IDE or the browser) has a compatible Python installation. Packages are installed only when missing or when the required Biogeme version differs. An internet-enabled kernel with permission to install packages is required for first-time use. When the installation finishes, continue running the remaining cells. If an import still reports a stale package, restart the kernel and run all cells once more.

In [ ]:
# Server-backed Jupyter / Google Colab setup. No terminal commands required.
import sys
import os
import subprocess
import importlib
from importlib import metadata as package_metadata

print('Python:', sys.version.split()[0])
print('Python executable:', sys.executable)
print('Kernel platform:', sys.platform)

if sys.platform in {'emscripten', 'wasi'}:
    raise RuntimeError(
        'This is a browser-only JupyterLite/Pyodide kernel. Biogeme needs a full '
        'Python kernel. Open this notebook in Google Colab or server-backed Jupyter.'
    )
if sys.version_info < (3, 12):
    raise RuntimeError(
        'Biogeme 3.3.5 requires Python 3.12 or newer. '
        'Select a newer Python kernel, or open the notebook in Google Colab.'
    )

# Only install packages that are missing. Avoid needlessly upgrading a hosted kernel.
required_modules = {
    'biogeme': 'biogeme==3.3.5',
    'pandas': 'pandas>=2.2',
    'numpy': 'numpy>=1.26',
    'scipy': 'scipy>=1.12',
    'matplotlib': 'matplotlib>=3.8',
    'openpyxl': 'openpyxl>=3.1',
}

needed = [spec for module, spec in required_modules.items()
          if importlib.util.find_spec(module) is None]
try:
    installed_biogeme = package_metadata.version('biogeme')
except package_metadata.PackageNotFoundError:
    installed_biogeme = None
if installed_biogeme is not None and installed_biogeme != '3.3.5':
    needed.append('biogeme==3.3.5')
needed = list(dict.fromkeys(needed))

if needed:
    print('Installing into the active Jupyter kernel:', ', '.join(needed))
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *needed])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Package installation failed in this hosted kernel. Try Google Colab '
            'with an internet-enabled Python 3.12+ runtime, or ask the JupyterHub '
            'administrator to enable/install Biogeme 3.3.5.'
        ) from exc
    importlib.invalidate_caches()
else:
    print('All required packages are already available. No installation needed.')

import biogeme
print('Biogeme version:', package_metadata.version('biogeme'))
print('Environment ready. Continue with the next cell.')

## 2. Workbook placement (no embedded data)

Place **`SP6ModeChoiceTLV.xlsx` in the same working folder as this notebook**. The notebook reads the Excel file directly with `pd.read_excel()`; no Base64 data, downloaded file, or hard-coded computer-specific path is used.

In a standard Jupyter Notebook/JupyterLab session, the current working folder is ordinarily the folder in which you opened the notebook. In Google Colab, place the Excel workbook in the session Files area (`/content`) before running it. The notebook prints its working directory and gives a helpful error if the workbook is missing.

**Example folder layout:**

```text
Choice_Exercise/
├── Biogeme_6Mode_Student_Jupyter_Same_Folder.ipynb
├── SP6ModeChoiceTLV.xlsx
└── CaseStudy1_MNL_TLV.docx   (optional reference)
```

## 3. Model settings and imports

For the basic classroom exercise, leave the settings unchanged. The time coefficient is shared; costs and status effects are allowed to differ by mode. After changing a setting, use **Run All** to avoid mixing old data with new model definitions.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import chi2

DATA_FILE = Path('SP6ModeChoiceTLV.xlsx')  # supplied in the Jupyter working folder
DATA_SHEET = 'dataset_work_trips'
OUTPUT_DIR = Path.cwd() / 'calibration_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Core identification and units
REFERENCE_ALT = 1              # BW: fixes its ASC and all person-level dummy effects to zero
TIME_DIVISOR = 10.0            # raw minutes / 10 => time coefficient per ten minutes
CENTS_PER_NIS = 100.0

# The spreadsheet provides NO observed monetary payment specifically for car passengers.
# 0.0 = no observed out-of-pocket passenger cost (recommended starting assumption).
# >0 = use the indicated share of cfuel as an exploratory cost proxy, not a measured fare.
CAR_PASSENGER_FUEL_SHARE = 0.0

# Add further socioeconomic dummy interactions if justified:
# The currently active dummy factors are work=2 and work=3.
DUMMY_DEFINITIONS = {
    'WORK2': ('work', 2),            # part-time versus full-time
    'WORK3': ('work', 3),            # working and studying versus full-time
    'FEMALE': ('sex', 2),            # optional
    'NO_LICENSE': ('drivlic', 2),    # optional; car-driver availability may be related
}
ACTIVE_DUMMIES = ['WORK2', 'WORK3']

# Omit an alt-specific dummy coefficient if its subgroup never selects the alternative.
# Set higher (e.g. 5 or 10) for more conservative sparse-cell protection.
MIN_SUBGROUP_CHOICES = 3

# Estimate the restricted models as well as the requested model for formal comparison.
MODEL_CONFIGS = {
    'M0_generic_cost': {'cost_specific': False, 'include_dummies': False},
    'M1_specific_cost': {'cost_specific': True, 'include_dummies': False},
    'M2_specific_cost_and_work': {'cost_specific': True, 'include_dummies': True},
}

ALT_NAMES = {1: 'BW', 2: 'BKR', 4: 'MTW', 5: 'MTKR', 8: 'CD', 9: 'CP'}
ALT_LABELS = {
    1: 'Bus: walk', 2: 'Bus: kiss-and-ride',
    4: 'Mass transit: walk', 5: 'Mass transit: kiss-and-ride',
    8: 'Car: driver', 9: 'Car: passenger',
}
ALT_IDS = list(ALT_NAMES)
print('Dataset location:', DATA_FILE.resolve())
print('Model folder:', OUTPUT_DIR.resolve())

## 4. Load, validate, and describe the actual spreadsheet

In [ ]:
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f'Cannot find {DATA_FILE.name!r} in the Jupyter working folder: {Path.cwd()}. '
        'Upload/copy the Excel workbook into this folder (same working folder as the notebook), '
        'then rerun this cell. In Colab, upload it to the /content Files area.'
    )
raw = pd.read_excel(DATA_FILE, sheet_name=DATA_SHEET)
dictionary = pd.read_excel(DATA_FILE, sheet_name='dictionary', header=None)
raw.columns = raw.columns.astype(str).str.strip()

NEEDED = {
    'rno','exper','choice','work','sex','drivlic',
    'binvt','bovt','bawt','badt','bfare',
    'minvt','movt','mawt','madt','mfare',
    'ctime','csrch','cwalk','cfuel','cpark',
    *[f'av{i}' for i in ALT_IDS],
}
missing = sorted(NEEDED.difference(raw.columns))
assert not missing, f'Required Excel columns are missing: {missing}'

# Biogeme requires a fully numeric data frame (no objects or NA in model columns).
for column in NEEDED:
    raw[column] = pd.to_numeric(raw[column], errors='raise')
if raw[list(NEEDED)].isna().any().any():
    raise ValueError('Model inputs contain missing values; inspect before estimation.')
if not raw['choice'].isin(ALT_IDS).all():
    raise ValueError('The file contains choices outside the six available alternatives.')
if not raw['work'].isin([1, 2, 3, 4]).all():
    raise ValueError('Unexpected work-status codes.')
for i in ALT_IDS:
    if not raw[f'av{i}'].isin([0, 1]).all():
        raise ValueError(f'av{i} must be binary (0/1).')
    bad = (raw['choice'] == i) & (raw[f'av{i}'] != 1)
    if bad.any():
        raise ValueError(f'{bad.sum()} rows choose unavailable alternative {i}.')
if raw[[f'av{i}' for i in ALT_IDS]].sum(axis=1).eq(0).any():
    raise ValueError('At least one record has no available alternatives.')

print(f'Observations: {len(raw):,} | Distinct respondents: {raw.rno.nunique():,}')
print('\nEmployment categories (1 full-time; 2 part-time; 3 work+study; 4 other):')
display(raw['work'].value_counts().sort_index().rename('Observations').to_frame())
print('\nChoice shares:')
choice_share = (raw.choice.value_counts(normalize=True).reindex(ALT_IDS, fill_value=0)
                .rename(index=ALT_LABELS).mul(100).round(2))
display(choice_share.rename('Observed share (%)').to_frame())
print('\nAvailability by alternative:')
display(pd.DataFrame({ALT_NAMES[i]: {
    'Available observations': int(raw[f'av{i}'].sum()),
    'Chosen observations': int((raw.choice == i).sum())}
    for i in ALT_IDS}).T)
print('\nOriginal-column numeric summary (selected):')
display(raw[['binvt','bovt','bfare','minvt','movt','mfare',
             'ctime','cfuel','cpark']].describe().round(2).T)

## 5. Construct travel time, cost, and employment dummies

**Time mapping (modelling assumption):**

- BW: `binvt + bovt + bawt`.
- BKR: `binvt + bovt + badt`.
- MTW: `minvt + movt + mawt`.
- MTKR: `minvt + movt + madt`.
- CD: `ctime + csrch + cwalk`.
- CP: `ctime` (not charged driver-only parking search or walk time).

**Cost mapping (modelling assumption):** BW/BKR use `bfare`, MTW/MTKR use `mfare`, CD uses `cfuel + cpark`, and CP uses `CAR_PASSENGER_FUEL_SHARE * cfuel`. The last case is a *proxy scenario*, not an observed passenger fare. With share zero, no passenger cost coefficient is estimable or included.

To modify the definitions, change only the `raw_time` / `raw_cost_cents` dictionaries below. The estimation code remains unchanged.

In [ ]:
raw_time = {
    1: raw.binvt + raw.bovt + raw.bawt,
    2: raw.binvt + raw.bovt + raw.badt,
    4: raw.minvt + raw.movt + raw.mawt,
    5: raw.minvt + raw.movt + raw.madt,
    8: raw.ctime + raw.csrch + raw.cwalk,
    9: raw.ctime,
}
raw_cost_cents = {
    1: raw.bfare,
    2: raw.bfare,
    4: raw.mfare,
    5: raw.mfare,
    8: raw.cfuel + raw.cpark,
    9: CAR_PASSENGER_FUEL_SHARE * raw.cfuel,
}

# Only model variables enter Biogeme, keeping its database unambiguously numeric.
data = raw[['rno','exper','choice'] + [f'av{i}' for i in ALT_IDS]].copy()
for i, code_name in ALT_NAMES.items():
    data[f'TT_{code_name}'] = raw_time[i].to_numpy(dtype=float) / TIME_DIVISOR
    data[f'COST_{code_name}'] = raw_cost_cents[i].to_numpy(dtype=float) / CENTS_PER_NIS

for dummy_name in ACTIVE_DUMMIES:
    source, positive_code = DUMMY_DEFINITIONS[dummy_name]
    data[dummy_name] = (raw[source] == positive_code).astype(int)
    if not data[dummy_name].any():
        raise ValueError(f'{dummy_name} has no observations. Remove it from ACTIVE_DUMMIES.')

assert np.isfinite(data.to_numpy(dtype=float)).all()
for i, name in ALT_NAMES.items():
    assert (data[f'TT_{name}'] > 0).all(), f'Nonpositive time for {name}.'
    assert (data[f'COST_{name}'] >= 0).all(), f'Negative cost for {name}.'

# Panel data must be grouped by individual ID.
data = data.sort_values(['rno','exper']).reset_index(drop=True)
print('Prepared modelling matrix:', data.shape)
display(data.head())

print('\nCost and time variability among available observations:')
variation = []
for i, name in ALT_NAMES.items():
    subset = data.loc[data[f'av{i}'] == 1]
    variation.append({
        'Alternative': name,
        'TT mean (10 min)': subset[f'TT_{name}'].mean(),
        'TT sd': subset[f'TT_{name}'].std(),
        'COST mean (NIS)': subset[f'COST_{name}'].mean(),
        'COST sd': subset[f'COST_{name}'].std(),
    })
display(pd.DataFrame(variation).round(3))

print('\nEmployment-status-by-choice counts:')
work_cross = pd.crosstab(raw['work'], raw['choice']).reindex(columns=ALT_IDS, fill_value=0)
work_cross.columns = [ALT_NAMES[i] for i in ALT_IDS]
display(work_cross)
if raw['work'].value_counts().min() < 50:
    print('CAUTION: Some employment categories have few observations. Inspect individual coefficients.')

## 6. Preliminary choice and variable plots

In [ ]:
import matplotlib.pyplot as plt  # plotting import (explicit for online Jupyter)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw['choice'].map(ALT_LABELS).value_counts().reindex(list(ALT_LABELS.values())).plot(
    kind='bar', ax=axes[0], title='Observed choices')
axes[0].set_ylabel('Number of SP observations')
axes[0].tick_params(axis='x', rotation=50)
work_cross.plot(kind='bar', stacked=True, ax=axes[1], title='Observed choices by work status')
axes[1].set_xlabel('Work status: 1 full-time; 2 part-time; 3 work+study')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Alternative', fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'descriptive_choice_counts.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Biogeme model factory and identification safeguards

We compare three **nested** models fitted to exactly the same observations:

- **M0:** five ASCs (BW reference), shared time coefficient, shared cost coefficient.
- **M1:** same ASCs and time coefficient, but alternative-specific cost coefficients (where cost varies).
- **M2 (requested):** M1 plus alternative-specific employment dummy coefficients for all *estimable* combinations.

For the last model, some dummy–alternative combinations are omitted if fewer than `MIN_SUBGROUP_CHOICES` respondents' choice records actually select the alternative while the dummy equals one. This avoids fitting near-infinite coefficients in obviously separated cells. These restrictions are printed explicitly. `MIN_SUBGROUP_CHOICES` can be changed with care.

The respondent-level panel likelihood is
\[
\log L(\boldsymbol\beta)=\sum_n\log\left[\prod_{t\in n}
P_{nt,y_{nt}}(\boldsymbol\beta)\right],\qquad
P_{nti}=\frac{a_{nti}\exp(V_{nti})}{\sum_j a_{ntj}\exp(V_{ntj})}.
\]

In [ ]:
import biogeme
from biogeme.biogeme import BIOGEME
from biogeme.database import Database
from biogeme.expressions import Beta, Variable, PanelLikelihoodTrajectory, log
from biogeme.models import logit
from biogeme.results_processing import get_pandas_estimated_parameters

print('Biogeme module:', biogeme.__file__)

def eligible_dummies(model_data):
    """Find subgroup×alternative cells with enough chosen observations."""
    chosen_counts = {}
    skipped = []
    for dummy_name in ACTIVE_DUMMIES:
        for alt_id, code_name in ALT_NAMES.items():
            if alt_id == REFERENCE_ALT:
                continue  # normalize all person-level dummy effects for reference alternative
            chosen = int(((model_data['choice'] == alt_id) & (model_data[dummy_name] == 1)).sum())
            if chosen < MIN_SUBGROUP_CHOICES:
                skipped.append((dummy_name, code_name, chosen))
            else:
                chosen_counts[(dummy_name, alt_id)] = chosen
    return chosen_counts, skipped

ELIGIBLE_DUMMY_CELLS, SKIPPED_DUMMY_CELLS = eligible_dummies(data)
print('Eligible alternative-specific dummy coefficients:', len(ELIGIBLE_DUMMY_CELLS))
print('Omitted dummy cells (dummy, alternative, chosen count):', SKIPPED_DUMMY_CELLS)
for dummy_name, alt_id in ELIGIBLE_DUMMY_CELLS:
    n_choice = ELIGIBLE_DUMMY_CELLS[(dummy_name, alt_id)]
    if n_choice < 10:
        warnings.warn(f'Sparse: {dummy_name} for {ALT_NAMES[alt_id]} has only {n_choice} chosen records.')

# No ambiguity about the sample: one row per stated-preference task, grouped by respondent.
biogeme_db = Database('TLV_six_modes', data.copy())
biogeme_db.panel('rno')

# Under uniform probability among available alternatives, null LL is row-wise log(1/K_available).
null_ll = -np.log(data[[f'av{i}' for i in ALT_IDS]].sum(axis=1).to_numpy()).sum()
print('Analytical null log likelihood:', float(null_ll))

def build_model(name, cost_specific=True, include_dummies=True):
    """Create a Biogeme panel-MNL with the SAME B_TIME for every mode."""
    b_time = Beta('B_TIME', 0, None, None, 0)
    b_generic_cost = None if cost_specific else Beta('B_COST', 0, None, None, 0)
    utilities = {}
    availabilities = {}
    for alt_id, code_name in ALT_NAMES.items():
        # IMPORTANT: this exact b_time object is reused in all utilities.
        utility = b_time * Variable(f'TT_{code_name}')
        if alt_id != REFERENCE_ALT:
            utility += Beta(f'ASC_{code_name}', 0, None, None, 0)

        cost_col = f'COST_{code_name}'
        cost_var = data.loc[data[f'av{alt_id}'] == 1, cost_col]
        if cost_var.std() > 1e-12:  # constant-zero CP price cannot identify a price slope
            if cost_specific:
                utility += Beta(f'B_COST_{code_name}', 0, None, None, 0) * Variable(cost_col)
            else:
                utility += b_generic_cost * Variable(cost_col)
        elif cost_var.abs().max() > 1e-12:
            print(f'Warning: {code_name} cost is constant and nonzero, confounded with ASC.')

        if include_dummies and alt_id != REFERENCE_ALT:
            for dummy_name in ACTIVE_DUMMIES:
                if (dummy_name, alt_id) in ELIGIBLE_DUMMY_CELLS:
                    utility += Beta(f'B_{dummy_name}_{code_name}', 0, None, None, 0) * Variable(dummy_name)

        utilities[alt_id] = utility
        availabilities[alt_id] = Variable(f'av{alt_id}')

    # PanelLikelihoodTrajectory combines all SP tasks of the same individual.
    per_task_probability = logit(utilities, availabilities, Variable('choice'))
    log_likelihood = log(PanelLikelihoodTrajectory(per_task_probability))
    engine = BIOGEME(
        biogeme_db, log_likelihood,
        generate_html=False, generate_yaml=False,
    )
    engine.model_name = name
    return engine

## 8. Maximum-likelihood estimation

Run this cell to calibrate M0, M1, and your requested M2 model. Estimated values, standard errors, test statistics, and p-values are written as CSVs. **Inspect convergence and coefficient magnitudes** before using any result. The workbook contains few part-time observations, so the fully interacted model may be unstable. Treat coefficients with extreme magnitudes or large standard errors cautiously.

In [ ]:
estimated = {}
parameter_tables = {}
model_summaries = []

for model_name, config in MODEL_CONFIGS.items():
    print(f'\n=== Estimating {model_name} ===')
    engine = build_model(model_name, **config)
    result = engine.estimate()
    estimated[model_name] = result
    print(result.short_summary())
    print('Optimizer converged:', result.algorithm_has_converged)

    tables = get_pandas_estimated_parameters(estimation_results=result)
    if isinstance(tables, dict):
        table = tables.get('Estimated parameters', next(iter(tables.values())))
    else:
        table = tables
    parameter_tables[model_name] = table.copy()
    display(table)
    table.to_csv(OUTPUT_DIR / f'{model_name}_parameters.csv', index=False)

    ll = float(result.raw_estimation_results.final_log_likelihood)
    k = len(result.get_beta_values())
    n_individuals = data.rno.nunique()
    model_summaries.append({
        'Model': model_name,
        'Parameters': k,
        'LL': ll,
        'AIC': 2*k - 2*ll,
        'BIC (respondents)': np.log(n_individuals)*k - 2*ll,
        'McFadden rho2': 1 - ll/null_ll,
        'Converged': bool(result.algorithm_has_converged),
    })

summary = pd.DataFrame(model_summaries)
print('\nComparison (lower AIC/BIC is better; inspect stability and interpretability):')
display(summary.round(4))
summary.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

## 9. Nested-model tests (likelihood-ratio tests)

In [ ]:
def likelihood_ratio_test(smaller, larger):
    a = summary.set_index('Model').loc[smaller]
    b = summary.set_index('Model').loc[larger]
    df_difference = int(b['Parameters'] - a['Parameters'])
    if df_difference <= 0:
        raise ValueError('The larger model must have more free parameters.')
    lr = 2.0 * (b['LL'] - a['LL'])
    return {
        'Restricted model': smaller,
        'Unrestricted model': larger,
        'LR statistic': lr,
        'df': df_difference,
        'p value': float(chi2.sf(max(0, lr), df_difference)),
    }

lr_table = pd.DataFrame([
    likelihood_ratio_test('M0_generic_cost', 'M1_specific_cost'),
    likelihood_ratio_test('M1_specific_cost', 'M2_specific_cost_and_work'),
    likelihood_ratio_test('M0_generic_cost', 'M2_specific_cost_and_work'),
])
display(lr_table)
lr_table.to_csv(OUTPUT_DIR / 'likelihood_ratio_tests.csv', index=False)

print('Interpretation: LR tests use nested hypotheses. If optimization did not converge,')
print('or coefficients lie near unidentified/separated regions, the asymptotic p-values may be unreliable.')

## 10. Predicted choice probabilities and observed-versus-predicted shares

For transparent post-estimation checking, probabilities are evaluated directly using the fitted coefficients. The sample log likelihood of these probabilities is compared against the Biogeme result. The two should agree up to numerical tolerance. The six probability columns always sum to one and unavailable modes always receive probability zero.

In [ ]:
from scipy.special import logsumexp

AID_INDEX = {alt_id: j for j, alt_id in enumerate(ALT_IDS)}

def probability_matrix(betas):
    u = np.zeros((len(data), len(ALT_IDS)), dtype=float)
    availability = data[[f'av{i}' for i in ALT_IDS]].to_numpy(dtype=bool)
    for j, (alt_id, code_name) in enumerate(ALT_NAMES.items()):
        u[:, j] = betas['B_TIME'] * data[f'TT_{code_name}'].to_numpy()
        u[:, j] += betas.get(f'ASC_{code_name}', 0.0)
        if 'B_COST' in betas:
            u[:, j] += betas['B_COST'] * data[f'COST_{code_name}'].to_numpy()
        else:
            u[:, j] += betas.get(f'B_COST_{code_name}', 0.0) * data[f'COST_{code_name}'].to_numpy()
        for dummy_name in ACTIVE_DUMMIES:
            u[:, j] += betas.get(f'B_{dummy_name}_{code_name}', 0.0) * data[dummy_name].to_numpy()
    u[~availability] = -np.inf
    logp = u - logsumexp(u, axis=1, keepdims=True)
    p = np.exp(logp)
    assert np.allclose(p.sum(axis=1), 1, atol=1e-10)
    assert np.all(p[~availability] == 0)
    chosen_idx = data.choice.map(AID_INDEX).to_numpy()
    prediction_ll = logp[np.arange(len(data)), chosen_idx].sum()
    return p, float(prediction_ll)

fitted_shares = []
for model_name, result in estimated.items():
    p, check_ll = probability_matrix(result.get_beta_values())
    biogeme_ll = float(result.raw_estimation_results.final_log_likelihood)
    ll_error = check_ll - biogeme_ll
    print(f'{model_name}: prediction LL = {check_ll:.5f}; Biogeme LL = {biogeme_ll:.5f}; difference = {ll_error:.4g}')
    if not np.isclose(check_ll, biogeme_ll, rtol=1e-5, atol=1e-2):
        warnings.warn('CHECK: predicted and Biogeme log likelihoods differ. Verify model specification.')
    pred_out = data[['rno','exper','choice']].copy()
    for j, aid in enumerate(ALT_IDS):
        pred_out[f'P_{ALT_NAMES[aid]}'] = p[:, j]
        fitted_shares.append({
            'Model': model_name,
            'Alternative': ALT_NAMES[aid],
            'Observed share': float((data.choice == aid).mean()),
            'Predicted share': float(p[:, j].mean()),
        })
    pred_out.to_csv(OUTPUT_DIR / f'{model_name}_predicted_probabilities.csv', index=False)

share_df = pd.DataFrame(fitted_shares)
display(share_df.pivot(index='Alternative', columns='Model', values='Predicted share').round(4))
final_model = 'M2_specific_cost_and_work'
share_final = share_df.query('Model == @final_model').set_index('Alternative')
share_final[['Observed share', 'Predicted share']].plot(kind='bar', figsize=(9, 4))
plt.title('Observed vs predicted mode shares (requested M2 model)')
plt.ylabel('Share of SP observations')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'observed_vs_predicted_shares.png', dpi=180, bbox_inches='tight')
plt.show()
share_df.to_csv(OUTPUT_DIR / 'observed_vs_predicted_shares.csv', index=False)

## 11. Interpret the common time coefficient and mode-specific cost trade-offs

Because `TT = minutes / TIME_DIVISOR`, the coefficient per raw **minute** is `B_TIME / TIME_DIVISOR`. Since costs are NIS, the value of time for mode \(i\) is

\[
VOT_i = 60\,\frac{\beta_{TT}/TIME\_DIVISOR}{\beta_{COST,i}}
\qquad\text{(NIS/hour)}.
\]

It is economically meaningful only when **both** coefficients are negative and the cost parameter is estimated. For the default zero car-passenger cost, no car-passenger value of time can be calculated. Common time preferences do *not* imply common values of time if the monetary-cost coefficients differ by mode.

In [ ]:
betas = estimated['M2_specific_cost_and_work'].get_beta_values()
common_time_per_10min = betas['B_TIME']
common_time_per_minute = common_time_per_10min / TIME_DIVISOR
print(f'B_TIME (per {TIME_DIVISOR:g} min): {common_time_per_10min:.6g}')
print(f'B_TIME (per minute): {common_time_per_minute:.6g}')

vot_rows = []
for alt_id, alt_name in ALT_NAMES.items():
    cost_name = f'B_COST_{alt_name}'
    b_cost = betas.get(cost_name)
    economically_valid = (b_cost is not None and b_cost < 0 and common_time_per_minute < 0)
    vot_rows.append({
        'Alternative': alt_name,
        'Cost coefficient (per NIS)': b_cost,
        'VOT (NIS/hour)': 60*common_time_per_minute / b_cost if economically_valid else np.nan,
        'Interpretation': ('Time and cost both negative' if economically_valid else
                           'Unavailable or non-negative coefficient; no valid VOT'),
    })
vot_table = pd.DataFrame(vot_rows)
display(vot_table.round(4))
vot_table.to_csv(OUTPUT_DIR / 'mode_specific_value_of_time.csv', index=False)

if common_time_per_minute >= 0:
    warnings.warn('The common travel-time coefficient is non-negative; check model specification and data.')
if any(x is not None and x >= 0 for x in [betas.get(f'B_COST_{code}') for code in ALT_NAMES.values()]):
    warnings.warn('Some estimated cost coefficients are non-negative; assess their significance and data coding.')

## 12. Export reproducibility information

The final results folder contains coefficient tables, model comparisons, likelihood-ratio tests, predicted probabilities, observed/predicted shares, a value-of-time table, and plots. The notebook file itself is the Biogeme estimation input. **Inspect** the fit and hypothesis tests before reporting the richest model as the final specification.

### Important research limitations

1. Individual respondents answer repeated SP tasks. The panel MNL above **does not introduce taste heterogeneity or serially correlated utility shocks**. It groups the likelihood by individual but does not make this a mixed logit.
2. Only 40 records are part-time workers and 100 are work-and-study respondents. Some group-by-alternative coefficients may be weakly identified and are skipped when separated.
3. Car-passenger price is not directly observed. A nonzero fuel share is a sensitivity assumption, **not a measured respondent payment**.
4. The access-time components and allocation of parking costs need to be validated against the survey's questionnaire before publication.
5. A model with extra alternative-specific terms can overfit. The LR comparison, AIC/BIC, sign checks and (for research use) a respondent-grouped holdout assessment should inform selection.

In [ ]:
metadata = {
    'source_workbook': DATA_FILE.name,
    'data_sheet': DATA_SHEET,
    'observations': len(data),
    'respondents': int(data.rno.nunique()),
    'choice_codes': ALT_NAMES,
    'reference_alternative': ALT_NAMES[REFERENCE_ALT],
    'common_time_coefficient': 'B_TIME',
    'time_units': f'{TIME_DIVISOR:g} minutes',
    'cost_units': 'NIS (100 cents = 1 NIS)',
    'car_passenger_fuel_share_assumption': CAR_PASSENGER_FUEL_SHARE,
    'active_dummies': ACTIVE_DUMMIES,
    'employment_reference': 'work=1 (full-time)',
    'min_subgroup_chosen_count': MIN_SUBGROUP_CHOICES,
    'omitted_employment_interactions': [list(x) for x in SKIPPED_DUMMY_CELLS],
    'model_configs': MODEL_CONFIGS,
    'final_model_beta_values': {name: float(value) for name, value in betas.items()},
}
with open(OUTPUT_DIR / 'calibration_metadata.json', 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)
print('Saved calibration outputs in:', OUTPUT_DIR.resolve())
print('\n'.join('  ' + p.name for p in sorted(OUTPUT_DIR.iterdir())))

## 13. Students: interpretation and submission checklist

1. Explain why there is **one** `B_TIME` but multiple cost coefficients.
2. For time and cost, compare estimated signs and significance with your a-priori hypotheses.
3. Report the two strongest specifications, likelihood-ratio comparison, and relevant AIC/BIC values. Discuss any convergence failures or weakly identified employment interactions.
4. Interpret the mode-specific value of travel time only where time and cost coefficients have economically meaningful signs.
5. Submit the completed notebook and a short report as required by the case-study instructions.

### How to get the results from an online environment

Run the next cell after estimation to create a single results ZIP. In Google Colab, it will offer a browser download. In other Jupyter notebooks, the generated ZIP appears in the working directory and can be downloaded using the file browser.

**Reproducibility:** Keep the Excel workbook alongside this notebook when sharing or rerunning the exercise. The workbook is a separate supporting file. No local IDE, hard-coded computer path, or downloaded Python script is required.


In [ ]:
# Optional final cell: bundle result tables and plots for easy student submission.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

result_dir = Path.cwd() / 'calibration_results'
if not result_dir.is_dir() or not list(result_dir.iterdir()):
    raise RuntimeError('No outputs yet. Run the estimation and results cells before creating the ZIP.')
archive_path = Path.cwd() / 'Biogeme_student_results.zip'
with ZipFile(archive_path, 'w', compression=ZIP_DEFLATED) as archive:
    for file in sorted(result_dir.rglob('*')):
        if file.is_file():
            archive.write(file, arcname=file.relative_to(Path.cwd()))
print('Results archive:', archive_path)

try:
    from google.colab import files
except ImportError:
    print('In JupyterHub/JupyterLab, download the ZIP from the file browser.')
else:
    files.download(str(archive_path))